In [19]:
pip install squarify

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
Note: you may need to restart the kernel to use updated packages.


In [10]:
import pandas as pd
import matplotlib.pyplot as plt
import squarify 
from matplotlib.animation import FuncAnimation
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from PIL import Image
from IPython.display import HTML

""" leer datos """
df = pd.read_csv('medallasporpais.csv', sep=';', encoding='latin-1')
ediciones = df['Edición'].unique()

def formatear_nombre_archivo(nombre_pais):
    """
    convierte nombre para ruta de archivos de imagenes por nombre formato Banderas/PpppPppp.png
    """
    palabras = nombre_pais.split()
    nombre_formateado = "".join(word.capitalize() for word in palabras)
    return f"Banderas/{nombre_formateado}.png"

def cargar_imagen(ruta):
    """ carga la imagen """
    try:
        img = Image.open(ruta).convert('RGB') 
        return img
    except FileNotFoundError:
        return Image.new('RGB', (10, 10), color=(200, 200, 200))

""" configuracion de gráfico"""
fig, (ax_tree, ax_flags) = plt.subplots(1, 2, figsize=(16, 8), 
                                        gridspec_kw={'width_ratios': [3, 1]})

def actualizar(i):
    """ dependiendo de la edicion de jjoo se muestran hasta 9 paises y su cantidad de oros y un espacio para otros."""
    ax_tree.clear()
    ax_flags.clear()
    ax_flags.axis('off')
    
    edicion_actual = ediciones[i]
    data_edicion = df[df['Edición'] == edicion_actual].copy()
    data_edicion = data_edicion.sort_values(by='Medallas de Oro', ascending=False)
    
    top_9 = data_edicion.head(9).copy()
    otros_df = data_edicion.iloc[9:]
    
    if not otros_df.empty:
        suma_otros = otros_df['Medallas de Oro'].sum()
        fila_otros = pd.DataFrame([{'País': 'Otros', 'Medallas de Oro': suma_otros}])
        final_plot_df = pd.concat([top_9, fila_otros]).sort_values(by='Medallas de Oro', ascending=False)
    else:
        final_plot_df = top_9

    labels = [f"{n}\n({v})" for n, v in zip(final_plot_df['País'], final_plot_df['Medallas de Oro'])]
    cmap = plt.cm.get_cmap('tab20')
    colors = [cmap(i) for i in range(len(final_plot_df))]
    
    squarify.plot(sizes=final_plot_df['Medallas de Oro'], label=labels, 
                  alpha=0.8, color=colors, ax=ax_tree, 
                  text_kwargs={'fontsize': 10, 'fontweight': 'bold', 'color': 'white'})
    
    ax_tree.set_title(f"Distribución de Oros: {edicion_actual}", fontsize=18, pad=15)
    ax_tree.axis('off')


    """ Mostrar top3 paises"""
    top_3_paises = top_9.head(3)
    posiciones_y = [2, 1, 0]
    
    for idx, (_, row) in enumerate(top_3_paises.iterrows()):
        ruta = formatear_nombre_archivo(row['País'])
        img = cargar_imagen(ruta)
        imagebox = OffsetImage(img, zoom=0.03)
        ab = AnnotationBbox(imagebox, (0.1, posiciones_y[idx]), frameon=False, box_alignment=(0, 0.5))
        ax_flags.add_artist(ab)
        ax_flags.text(0.5, posiciones_y[idx], f"{row['País']}\n{row['Medallas de Oro']} Oros", 
                      va='center', ha='left', fontweight='bold')

    ax_flags.set_ylim(-0.5, 2.5)
    ax_flags.set_xlim(0, 1)
    ax_flags.set_title("LÍDERES", fontsize=12)

"animación para poder desplegar distintas ediciones."
ani = FuncAnimation(fig, actualizar, frames=len(ediciones), interval=2500)
plt.close()
HTML(ani.to_jshtml())